# 03 — Pitch Estimation Analysis

Visualises all three experimental conditions (A / B / C) across both domains (Western / Carnatic).

**Run the experiments first** (see `TUTORIAL.md §4`), then run this notebook top-to-bottom.
Cells gracefully skip if result files are not yet present.

| Condition | Model | Expected results dir |
|-----------|-------|----------------------|
| A | CREPE (Western pretrained, no fine-tuning) | `results/pitch/A_crepe/` |
| B | CREPELike trained from scratch on Saraga | `results/pitch/B_carnatic_from_scratch/` |
| C | CREPELike fine-tuned from CREPE on Saraga | `results/pitch/C_finetuned_from_crepe/` |

In [ ]:
import sys, os, json
from pathlib import Path

sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
RESULTS = Path('../results/pitch')

CONDITION_DIRS = {
    'A (CREPE)':         RESULTS / 'A_crepe',
    'B (from scratch)':  RESULTS / 'B_carnatic_from_scratch',
    'C (fine-tuned)':    RESULTS / 'C_finetuned_from_crepe',
}
CONDITION_COLORS = {
    'A (CREPE)':        '#4C72B0',
    'B (from scratch)': '#DD8452',
    'C (fine-tuned)':   '#55A868',
}

def load_json(path):
    with open(path) as f:
        return json.load(f)

def file_ok(path):
    return Path(path).exists() and Path(path).stat().st_size > 0

print('Results root:', RESULTS.resolve())
for cond, d in CONDITION_DIRS.items():
    print(f'  {cond}: {"OK" if d.exists() else "MISSING"}')

In [ ]:

# Quick results overview — works with partial results (any subset of A/B/C)
import json
from pathlib import Path
import pandas as pd

RESULTS = Path('../results/pitch')
CONDITION_DIRS = {
    'A (CREPE)':         RESULTS / 'A_crepe',
    'B (from scratch)':  RESULTS / 'B_carnatic_from_scratch',
    'C (fine-tuned)':    RESULTS / 'C_finetuned_from_crepe',
}

rows = []
for cond, d in CONDITION_DIRS.items():
    summary_path = d / 'pitch_summary.json'
    csv_path = d / 'pitch_results.csv'
    if summary_path.exists():
        with open(summary_path) as f:
            s = json.load(f)
        # also get domain breakdown from csv if available
        if csv_path.exists():
            df = pd.read_csv(csv_path).dropna(subset=['raw_pitch_accuracy'])
            for domain, grp in df.groupby('domain'):
                rows.append({
                    'condition': cond,
                    'domain': domain,
                    'n_tracks': len(grp),
                    'RPA': grp['raw_pitch_accuracy'].mean().round(3),
                    'OA': grp['overall_accuracy'].mean().round(3),
                    'MAE_cents': grp['mean_abs_error_cents'].mean().round(1),
                    'snap_ratio': grp['semitone_snap_ratio'].mean().round(3),
                })
        else:
            rows.append({
                'condition': cond,
                'domain': 'all',
                'n_tracks': s.get('raw_pitch_accuracy', {}).get('n', '?'),
                'RPA': round(s.get('raw_pitch_accuracy', {}).get('mean', float('nan')), 3),
                'OA': round(s.get('overall_accuracy', {}).get('mean', float('nan')), 3),
                'MAE_cents': round(s.get('mean_abs_error_cents', {}).get('mean', float('nan')), 1),
                'snap_ratio': round(s.get('semitone_snap_ratio', {}).get('mean', float('nan')), 3),
            })

if rows:
    overview = pd.DataFrame(rows).set_index(['condition', 'domain'])
    print("=== Results available so far ===")
    display(overview)
else:
    print("No results found yet — run bash run_pitch.sh first")


---
## 1 · Condition A — CREPE baseline (Western pretrained)
This is the off-the-shelf CREPE model run on both MAESTRO (Western) and Saraga (Carnatic).
The gap between domains is the *domain gap* the rest of the project aims to close.

In [ ]:
a_csv = CONDITION_DIRS['A (CREPE)'] / 'pitch_results.csv'
if not file_ok(a_csv):
    print('Condition A results not found — run §4.1 first')
else:
    df_a = pd.read_csv(a_csv)
    print(f'Condition A: {len(df_a)} tracks  |  domains: {df_a["domain"].value_counts().to_dict()}')
    display(df_a.groupby('domain')[['raw_pitch_accuracy','overall_accuracy','mean_abs_error_cents','semitone_snap_ratio']].mean().round(3))

In [ ]:
if file_ok(a_csv):
    metrics = [
        ('raw_pitch_accuracy',    'Raw Pitch Accuracy (RPA)'),
        ('overall_accuracy',      'Overall Accuracy (OA)'),
        ('mean_abs_error_cents',  'Mean |Error| (cents)'),
        ('semitone_snap_ratio',   'Semitone-Snap Ratio'),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (col, label) in zip(axes, metrics):
        sns.boxplot(data=df_a, x='domain', y=col, ax=ax,
                    order=['western','carnatic'],
                    palette=['#4C72B0','#DD8452'])
        ax.set_title(label, fontsize=10)
        ax.set_xlabel('')
    fig.suptitle('Condition A — CREPE (Western pretrained) on both domains', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(CONDITION_DIRS['A (CREPE)'] / 'figures' / 'condition_A_overview.png',
                dpi=150, bbox_inches='tight')
    plt.show()

### 1.1 · Semitone-snapping bias
CREPE was trained on Western (equal-temperament) music and tends to snap predicted frequencies toward the nearest semitone.
Carnatic gamakas span multiple microtones continuously — so a high `semitone_snap_ratio` on Carnatic data reveals this bias.

In [ ]:
if file_ok(a_csv):
    fig, ax = plt.subplots(figsize=(6, 4))
    means = df_a.groupby('domain')['semitone_snap_ratio'].mean()
    errs  = df_a.groupby('domain')['semitone_snap_ratio'].std()
    ax.bar(means.index, means.values, yerr=errs.values, capsize=5,
           color=['#4C72B0','#DD8452'], alpha=0.85)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Semitone-snap ratio')
    ax.set_title('CREPE semitone-snapping bias per domain\n(high = model pulls pitch toward ET grid)')
    plt.tight_layout()
    plt.show()

### 1.2 · Gamaka analysis — CREPE on Carnatic

In [ ]:
gamaka_path = CONDITION_DIRS['A (CREPE)'] / 'gamaka_errors.csv'
if not file_ok(gamaka_path):
    print('gamaka_errors.csv not found — run §4.1 first')
else:
    gdf = pd.read_csv(gamaka_path)
    melted = gdf.melt(id_vars='track_id',
                      value_vars=['mae_gamaka','mae_non_gamaka'],
                      var_name='region', value_name='MAE (cents)')
    melted['region'] = melted['region'].map({'mae_gamaka':'Gamaka','mae_non_gamaka':'Non-gamaka'})

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.boxplot(data=melted, x='region', y='MAE (cents)', ax=ax,
                palette=['#DD8452','#4C72B0'])
    ax.set_title('CREPE pitch error: gamaka vs stable regions\n(Carnatic test set)')
    plt.tight_layout()
    plt.savefig(CONDITION_DIRS['A (CREPE)'] / 'figures' / 'gamaka_vs_non_gamaka_A.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(gdf[['mae_gamaka','mae_non_gamaka']].describe().round(1))

---
## 2 · Learning curves — Conditions B and C
B = from-scratch training on Saraga.  C = fine-tuning from CREPE weights.
Overlay them to see whether Western pretraining gives a better starting point.

In [ ]:
def load_history(cond_name):
    p = CONDITION_DIRS[cond_name] / 'history.json'
    if not file_ok(p):
        return None
    h = load_json(p)
    return pd.DataFrame(h)

hist_b = load_history('B (from scratch)')
hist_c = load_history('C (fine-tuned)')

if hist_b is None and hist_c is None:
    print('No training history found — run §4.2 and §4.3 first')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, ylabel in zip(
        axes,
        ['val_rpa', 'val_loss'],
        ['Val RPA (Raw Pitch Accuracy)', 'Val Loss']
    ):
        for hist, label, color in [
            (hist_b, 'B (from scratch)', '#DD8452'),
            (hist_c, 'C (fine-tuned)',   '#55A868'),
        ]:
            if hist is not None and metric in hist.columns:
                ax.plot(hist[metric], label=label, color=color, lw=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.set_title(ylabel)
    fig.suptitle('Training curves — B (from scratch) vs C (fine-tuned from CREPE)', y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS / 'learning_curves_B_vs_C.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 3 · Three-condition comparison — the 3 × 2 grid
Combines `pitch_results.csv` (condition A) and `comparison_all.csv` (conditions B and C)
into a single long-form DataFrame, then plots every metric across all conditions and domains.

In [ ]:
frames = []

# Condition A
if file_ok(a_csv):
    df_tmp = pd.read_csv(a_csv).copy()
    df_tmp['condition'] = 'A'
    frames.append(df_tmp)

# Conditions B and C — 'comparison_all.csv' has all methods per track
for cond_label, cond_key in [('B', 'B (from scratch)'), ('C', 'C (fine-tuned)')]:
    p = CONDITION_DIRS[cond_key] / 'comparison_all.csv'
    if file_ok(p):
        df_tmp = pd.read_csv(p)
        # Keep only the trained carnatic model rows (not CREPE/pyin baselines)
        df_tmp = df_tmp[df_tmp['method'] == 'carnatic'].copy()
        df_tmp['condition'] = cond_label
        frames.append(df_tmp)

if not frames:
    print('No results found yet — run §4 first')
else:
    combined = pd.concat(frames, ignore_index=True)
    print(f'Combined: {len(combined)} rows  |  conditions: {combined["condition"].unique()}')
    pivot = combined.groupby(['condition','domain'])[[
        'raw_pitch_accuracy','overall_accuracy','mean_abs_error_cents','semitone_snap_ratio'
    ]].mean().round(3)
    display(pivot)

In [ ]:
if frames:
    METRICS = [
        ('raw_pitch_accuracy',   'RPA'),
        ('overall_accuracy',     'OA'),
        ('mean_abs_error_cents', 'MAE (cents)'),
        ('semitone_snap_ratio',  'Semitone-snap'),
    ]
    DOMAINS   = ['western', 'carnatic']
    COND_ORDER= ['A', 'B', 'C']
    PALETTE   = ['#4C72B0','#DD8452','#55A868']

    fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey='row')

    for col, (metric, label) in enumerate(METRICS):
        for row, domain in enumerate(DOMAINS):
            ax = axes[row][col]
            sub = combined[combined['domain'] == domain]
            present = [c for c in COND_ORDER if c in sub['condition'].unique()]
            means = sub.groupby('condition')[metric].mean().reindex(present)
            errs  = sub.groupby('condition')[metric].std().reindex(present)
            colors = [PALETTE[COND_ORDER.index(c)] for c in present]
            ax.bar(present, means.values, yerr=errs.values, capsize=5,
                   color=colors, alpha=0.85)
            ax.set_title(f'{label}\n({domain})', fontsize=9)
            ax.set_xlabel('')
            if col == 0:
                ax.set_ylabel(domain.capitalize())

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#4C72B0', label='A — CREPE (Western pretrained)'),
        Patch(facecolor='#DD8452', label='B — From scratch on Saraga'),
        Patch(facecolor='#55A868', label='C — Fine-tuned from CREPE'),
    ]
    fig.legend(handles=legend_elements, loc='upper center',
               ncol=3, fontsize=10, bbox_to_anchor=(0.5, 1.03))
    fig.suptitle('Pitch estimation — all conditions × all domains', y=1.06, fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS / 'condition_comparison_grid.png', dpi=150, bbox_inches='tight')
    plt.show()

### 3.1 · RPA focused: Western pretraining benefit on Carnatic?
The key scientific question: does fine-tuning from CREPE (C) beat training from scratch (B) on Carnatic data?
- **C > B** → Western pretraining transfers; low-level spectral features are culture-agnostic.
- **B > C** → Western inductive bias hurts Carnatic modelling; culturally-specific training is better.

In [ ]:
if frames:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, domain in zip(axes, ['western', 'carnatic']):
        sub = combined[combined['domain'] == domain]
        present = [c for c in COND_ORDER if c in sub['condition'].unique()]
        means = sub.groupby('condition')['raw_pitch_accuracy'].mean().reindex(present)
        errs  = sub.groupby('condition')['raw_pitch_accuracy'].std().reindex(present)
        colors = [PALETTE[COND_ORDER.index(c)] for c in present]
        bars = ax.bar(present, means.values, yerr=errs.values, capsize=5,
                      color=colors, alpha=0.85, width=0.5)
        for bar, val in zip(bars, means.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('Raw Pitch Accuracy (RPA)')
        ax.set_title(f'RPA — {domain.capitalize()} domain')
    fig.suptitle('Does Western pretraining help on Carnatic?  (C > B = yes)', fontsize=11)
    plt.tight_layout()
    plt.savefig(RESULTS / 'rpa_by_condition_and_domain.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 4 · Semitone-snapping bias across all conditions
Does fine-tuning on Carnatic data reduce the ET-snapping inherited from CREPE?
We expect: A (high snap) > C > B (lowest snap, no Western bias at all).

In [ ]:
if frames:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, domain in zip(axes, ['western', 'carnatic']):
        sub = combined[combined['domain'] == domain]
        present = [c for c in COND_ORDER if c in sub['condition'].unique()]
        means = sub.groupby('condition')['semitone_snap_ratio'].mean().reindex(present)
        errs  = sub.groupby('condition')['semitone_snap_ratio'].std().reindex(present)
        colors = [PALETTE[COND_ORDER.index(c)] for c in present]
        ax.bar(present, means.values, yerr=errs.values, capsize=5,
               color=colors, alpha=0.85, width=0.5)
        ax.set_ylim(0, 1)
        ax.set_ylabel('Semitone-snap ratio')
        ax.set_title(f'{domain.capitalize()} — semitone-snap')
    fig.suptitle('ET-snapping bias: does Carnatic training reduce it?', fontsize=11)
    plt.tight_layout()
    plt.savefig(RESULTS / 'semitone_snap_by_condition.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 5 · Gamaka analysis across conditions
Compare pitch MAE in gamaka vs stable regions, for each condition that has `gamaka_errors.csv`.

In [ ]:
gamaka_frames = []
for cond_label, cond_key in [
    ('A', 'A (CREPE)'),
    ('B', 'B (from scratch)'),
    ('C', 'C (fine-tuned)'),
]:
    p = CONDITION_DIRS[cond_key] / 'gamaka_errors.csv'
    if file_ok(p):
        g = pd.read_csv(p)
        g['condition'] = cond_label
        gamaka_frames.append(g)

if not gamaka_frames:
    print('No gamaka_errors.csv found — run §4 first')
else:
    gdf_all = pd.concat(gamaka_frames, ignore_index=True)
    melted = gdf_all.melt(
        id_vars=['track_id','condition'],
        value_vars=['mae_gamaka','mae_non_gamaka'],
        var_name='region', value_name='MAE (cents)'
    )
    melted['region'] = melted['region'].map({'mae_gamaka':'Gamaka','mae_non_gamaka':'Stable'})

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=melted, x='condition', y='MAE (cents)', hue='region',
                order=COND_ORDER, ax=ax,
                palette={'Gamaka':'#DD8452','Stable':'#4C72B0'})
    ax.set_title('Gamaka vs stable-pitch MAE by condition (Carnatic test set)')
    ax.set_xlabel('Condition')
    plt.tight_layout()
    plt.savefig(RESULTS / 'gamaka_analysis_by_condition.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 6 · Per-track scatter — domain gap visualisation

In [ ]:
if frames:
    # Scatter: RPA vs MAE (cents), colour = domain, marker = condition
    markers = {'A': 'o', 'B': 's', 'C': '^'}
    domain_colors = {'western': '#4C72B0', 'carnatic': '#DD8452'}

    fig, ax = plt.subplots(figsize=(8, 5))
    for (cond, domain), grp in combined.groupby(['condition','domain']):
        ax.scatter(
            grp['mean_abs_error_cents'],
            grp['raw_pitch_accuracy'],
            marker=markers.get(cond, 'o'),
            c=domain_colors[domain],
            alpha=0.6, s=40,
            label=f'{cond} / {domain}',
        )
    ax.set_xlabel('MAE (cents)')
    ax.set_ylabel('Raw Pitch Accuracy (RPA)')
    ax.set_title('Per-track RPA vs MAE — all conditions and domains')
    ax.legend(fontsize=8, ncol=2, loc='lower left')
    plt.tight_layout()
    plt.savefig(RESULTS / 'scatter_rpa_vs_mae.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 7 · Full summary table

In [ ]:
if frames:
    summary = combined.groupby(['condition','domain'])[[
        'raw_pitch_accuracy', 'raw_chroma_accuracy',
        'overall_accuracy', 'mean_abs_error_cents',
        'median_abs_error_cents', 'semitone_snap_ratio',
    ]].agg(['mean','std']).round(3)

    # Flatten column multi-index for display
    summary.columns = ['_'.join(c) for c in summary.columns]
    display(summary)

    # Save for cross-domain aggregation
    combined.to_csv(RESULTS / 'pitch_all_conditions.csv', index=False)
    print('\nSaved pitch_all_conditions.csv')

---
## 8 · Pitch contour examples
Visual spot-check: reference vs predicted F0 on one Carnatic track per condition.

In [ ]:
# Check for saved contour figures from run_pitch_experiments.py
found_any = False
for cond_label, cond_key in [
    ('A', 'A (CREPE)'),
    ('B', 'B (from scratch)'),
    ('C', 'C (fine-tuned)'),
]:
    fig_dir = CONDITION_DIRS[cond_key] / 'figures'
    if fig_dir.exists():
        contours = sorted(fig_dir.glob('contour_*.png'))
        if contours:
            found_any = True
            # Show first contour for this condition
            from IPython.display import Image, display as ipy_display
            print(f'Condition {cond_label} — {contours[0].name}')
            ipy_display(Image(filename=str(contours[0]), width=900))

if not found_any:
    print('No contour figures found — run §4 first')